## Trinity Challenge Stage 2 Submission

Dataset used:
emission-spot-primary-market-auction-report-2025-data.xlsx
<br>
Downloaded directly from the EEX EUA website [Here](https://www.eex.com/en/market-data/market-data-hub/environmentals/eex-eua-primary-auction-spot-download)


In [4]:
# Setup
! pip install dimod dwave-neal numpy pandas openpyxl matplotlib pulp


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


- `dimod` + `dwave-neal`: QUBO construction and SA solver
- `numpy` + `pandas`: data handling and the EEX xlsx
- `openpyxl`: reading the `.xlsx` file with pandas
- `matplotlib`: plots for the notebook (price distribution, cover ratio, energy convergence)
- `pulp`: IP baseline solver (the classical comparison the judges require)

In [13]:
# Quick sanity check to confirm all the imports
import dimod
import neal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pulp

print("dimod:", dimod.__version__)
print("neal:", neal.__version__)
print("pulp:", pulp.__version__)
print(pulp.listSolvers(onlyAvailable=True))
# Should show ['PULP_CBC_CMD'] at minimum

# Quick sanity check: tiny 2-variable QUBO
bqm = dimod.BinaryQuadraticModel(vartype='BINARY')
bqm.add_variable('x0', -1.0)  # wants to be 1
bqm.add_variable('x1', -1.0)  # wants to be 1
bqm.add_interaction('x0', 'x1', 3.0)  # but penalises both being 1
# E=−x0​−x1​+3x0​x1​ is the model

sampler = neal.SimulatedAnnealingSampler()
result = sampler.sample(bqm, num_reads=100)
print("\nBest sample:", result.first.sample)
print("Energy:", result.first.energy)
# Should give x0=1, x1=0 (or x1=1, x0=0) with energy -1.0

dimod: 0.12.21
neal: 0.6.0
pulp: 3.3.2
['PULP_CBC_CMD']

Best sample: {'x0': np.int8(1), 'x1': np.int8(0)}
Energy: -1.0


In [9]:

# Load
df_raw = pd.read_excel('emission-spot-primary-market-auction-report-2025-data.xlsx', header=5)
df_raw = df_raw.dropna(subset=['Date'])

# Keep only useful columns
keep = {
    'Date': 'date',
    'Auction Name': 'auction_type',
    'Auction Price €/tCO2': 'clearing_price',
    'Minimum Bid €/tCO2': 'min_bid',
    'Maximum Bid €/tCO2': 'max_bid',
    'Mean €/tCO2': 'mean_bid',
    'Median €/tCO2': 'median_bid',
    'Auction Volume tCO2': 'supply',
    'Number of bids submitted': 'n_bids',
    'Number of successful bids': 'n_successful',
    'Average bid size': 'avg_bid_size',
    'Standard deviation of bid volume per bidder': 'std_bid_volume',
    'Cover Ratio': 'cover_ratio',
    'Total Number of Bidders': 'n_bidders',
    'Number of Successful Bidders': 'n_successful_bidders',
    'Innovation Fund\n(IF)': 'fund_IF',
    'InnoFund RRF\n(IX)': 'fund_IX',
    'Modernisation Fund\n(MF)': 'fund_MF',
    'MS RRF\n(MX)': 'fund_MX',
    'Social Climate Fund\n(SF)': 'fund_SF',
}

df = df_raw[list(keep.keys())].rename(columns=keep).copy()
df['date'] = pd.to_datetime(df['date'])

# Clean auction type labels
type_map = {
    'Auction 4. Period CAP3 EU': 'EUA',
    'Auction 4. Period DE': 'EUA_DE',
    'Auction 4. Period CAP3 PL': 'EUA_PL',
    'Auction 4. Period CAP3 NIR': 'EUAA',
}
df['auction_type'] = df['auction_type'].map(type_map).fillna(df['auction_type'])

# Summary per type
print(df.groupby('auction_type')[['clearing_price','supply','n_bids','cover_ratio','avg_bid_size','std_bid_volume']].mean().round(2))
print(f"\nTotal sessions: {len(df)}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

print(df_raw['Auction Name'].value_counts())
print(f"\nTotal: {df_raw['Auction Name'].nunique()} unique types")

print(df)

/home/username/Desktop/Python/Quantum_Dice/lib64/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


              clearing_price      supply  n_bids  cover_ratio  avg_bid_size  \
auction_type                                                                  
EUA                    73.36  3252834.51   98.77         1.54      51159.30   
EUAA                   77.39   796500.00   49.00         1.65      26776.00   
EUA_DE                 73.78  1633411.11   86.64         2.06      39002.20   
EUA_PL                 73.25  2101300.00   94.08         1.79      40223.88   

              std_bid_volume  
auction_type                  
EUA                258434.39  
EUAA                70230.00  
EUA_DE             171873.29  
EUA_PL             184609.36  

Total sessions: 213
Date range: 2025-01-07 to 2025-12-15
Auction Name
Auction 4. Period CAP3 EU     142
Auction 4. Period DE           45
Auction 4. Period CAP3 PL      25
Auction 4. Period CAP3 NIR      1
Name: count, dtype: int64

Total: 4 unique types
          date auction_type  clearing_price  min_bid  max_bid  mean_bid  \
0   2025

## Data Limitations
while the data for all the conducted auctions is publicily available the exact bid made by each party in the bidding is not public information we solve this by building our own bid constructor which generates bids based on the meadian and mean data which is available

In [10]:
def reconstruct_bids(session, seed=42):
    rng = np.random.default_rng(seed)
    
    n         = int(session['n_bids'])
    mu_q      = session['avg_bid_size']
    sigma_q   = session['std_bid_volume']
    p_clear   = session['clearing_price']
    p_min     = session['min_bid']
    p_max     = session['max_bid']
    supply    = session['supply']
    tau       = session['auction_type']

    # --- Quantities ---
    # Log-normal parameterised directly from mean and std
    # If X ~ LogNormal(mu, sigma), then:
    #   E[X] = exp(mu + sigma^2/2) = mu_q
    #   Var[X] = (exp(sigma^2)-1)*exp(2*mu+sigma^2) = sigma_q^2
    var   = sigma_q**2
    mu_ln = np.log(mu_q**2 / np.sqrt(var + mu_q**2))
    sg_ln = np.sqrt(np.log(1 + var / mu_q**2))
    quantities = rng.lognormal(mu_ln, sg_ln, n)

    # --- Prices (willingness to pay) ---
    # Triangular distribution between min and max, peak at clearing price
    # Directly uses real data bounds — no free parameters
    prices = rng.triangular(p_min, p_clear, p_max, n)

    # --- Assemble bids ---
    bids = pd.DataFrame({
        'quantity': quantities,
        'price':    prices,
        'value':    quantities * prices,   # total € value — this is v_i
        'type':     tau,
        'supply':   supply,
    })

    return bids

# First define sessions dict
sessions = {
    tau: df[df['auction_type'] == tau].iloc[len(df[df['auction_type'] == tau])//2]
    for tau in ['EUA', 'EUA_DE', 'EUA_PL', 'EUAA']
}

# Then call on one session
s = sessions['EUA']
bids = reconstruct_bids(s)

print(f"Generated {len(bids)} bids")
print(f"Real avg_bid_size: {s['avg_bid_size']:,.0f}  |  Reconstructed: {bids['quantity'].mean():,.0f}")
print(f"Real std:          {s['std_bid_volume']:,.0f}  |  Reconstructed: {bids['quantity'].std():,.0f}")
print(f"Real min_bid:      {s['min_bid']:.2f}         |  Reconstructed: {bids['price'].min():.2f}")
print(f"Real max_bid:      {s['max_bid']:.2f}         |  Reconstructed: {bids['price'].max():.2f}")
print(bids.head())



Generated 100 bids
Real avg_bid_size: 45,115  |  Reconstructed: 20,462
Real std:          221,088  |  Reconstructed: 45,171
Real min_bid:      66.55         |  Reconstructed: 67.40
Real max_bid:      80.00         |  Reconstructed: 78.94
       quantity      price         value type   supply
0  15583.650867  69.582908  1.084356e+06  EUA  3245500
1   1395.752902  77.865472  1.086810e+05  EUA  3245500
2  34674.345522  74.281255  2.575654e+06  EUA  3245500
3  48770.202013  73.530996  3.586122e+06  EUA  3245500
4    272.188922  70.978198  1.931948e+04  EUA  3245500


In [11]:
def build_qubo(bids, A=1.0):
    """
    Build QUBO matrix Q such that x^T Q x is minimised at optimal allocation.
    
    Variables: x_i in {0,1}, one per bid
    H = H_obj + H_excl + H_cap
    """
    n = len(bids)
    bids = bids.reset_index(drop=True)
    
    # Penalty coefficients from Stage 1 dominance conditions
    v_max = bids['value'].max()
    B = A * (2 * v_max + 1)   # lot exclusivity: B > 2A * v_max
    C = A * (v_max + 1)        # per-type cap:    C > A * v_max

    Q = np.zeros((n, n))

    # H_obj: diagonal terms -A * v_i
    for i in range(n):
        Q[i, i] -= A * bids.loc[i, 'value']

    # H_excl: bids of the same type conflict (share the same lot pool)
    # penalise pairs (i,k) where tau(i) == tau(k)
    for i in range(n):
        for k in range(i+1, n):
            if bids.loc[i, 'type'] == bids.loc[k, 'type']:
                Q[i, k] += B  # upper triangle only

    # H_cap: per-type supply constraint
    # C * (sum_i q_i x_i - S_tau)^2 expanded:
    # C * sum_i q_i^2 x_i^2 + 2C * sum_{i<k} q_i q_k x_i x_k - 2C*S_tau * sum_i q_i x_i
    types = bids['type'].unique()
    for tau in types:
        mask = bids['type'] == tau
        idx  = bids[mask].index.tolist()
        S    = bids.loc[idx[0], 'supply']
        
        for i in idx:
            q_i = bids.loc[i, 'quantity']
            Q[i, i] += C * q_i**2 - 2 * C * S * q_i  # diagonal
            for k in idx:
                if k > i:
                    q_k = bids.loc[k, 'quantity']
                    Q[i, k] += 2 * C * q_i * q_k      # upper triangle

    return Q, {'A': A, 'B': B, 'C': C, 'n_bids': n, 'v_max': v_max}


# Build on EUA session
Q, params = build_qubo(bids)
print("QUBO shape:", Q.shape)
print("Penalty coefficients:")
for k,v in params.items():
    print(f"  {k}: {v:.4g}")
print(f"\nQ diagonal range: [{Q.diagonal().min():.4g}, {Q.diagonal().max():.4g}]")
print(f"Q off-diagonal nonzeros: {np.count_nonzero(Q - np.diag(Q.diagonal()))}")

QUBO shape: (100, 100)
Penalty coefficients:
  A: 1
  B: 6.071e+07
  C: 3.035e+07
  n_bids: 100
  v_max: 3.035e+07

Q diagonal range: [-7.754e+19, -5.363e+16]
Q off-diagonal nonzeros: 4950


In [14]:

def solve_sa(Q, bids, num_reads=1000, num_sweeps=2000):
    n = len(bids)
    bids = bids.reset_index(drop=True)
    
    # Load QUBO into dimod BQM
    labels = list(range(n))
    bqm = dimod.BinaryQuadraticModel(vartype='BINARY')
    
    for i in range(n):
        bqm.add_variable(i, Q[i, i])
    for i in range(n):
        for k in range(i+1, n):
            if Q[i, k] != 0:
                bqm.add_interaction(i, k, Q[i, k])
    
    # Run SA
    sampler = neal.SimulatedAnnealingSampler()
    result  = sampler.sample(bqm, num_reads=num_reads, num_sweeps=num_sweeps)
    
    # Best sample
    best = result.first.sample
    x    = np.array([best[i] for i in range(n)])
    
    # Decode results
    accepted       = bids[x == 1].copy()
    total_welfare  = accepted['value'].sum()
    total_volume   = accepted['quantity'].sum()
    supply         = bids['supply'].iloc[0]
    n_accepted     = int(x.sum())
    feasible       = total_volume <= supply
    
    return {
        'x':             x,
        'accepted':      accepted,
        'welfare':       total_welfare,
        'volume_used':   total_volume,
        'supply':        supply,
        'n_accepted':    n_accepted,
        'feasible':      feasible,
        'energy':        result.first.energy,
    }

sa_result = solve_sa(Q, bids)
print(f"Feasible:      {sa_result['feasible']}")
print(f"Accepted bids: {sa_result['n_accepted']} / {len(bids)}")
print(f"Volume used:   {sa_result['volume_used']:,.0f} / {sa_result['supply']:,.0f} tCO2")
print(f"SA Welfare:    €{sa_result['welfare']:,.2f}")
print(f"Energy:        {sa_result['energy']:.4g}")

Feasible:      True
Accepted bids: 100 / 100
Volume used:   2,046,151 / 3,245,500 tCO2
SA Welfare:    €148,040,004.01
Energy:        -2.761e+20
